# Standalone Teacher-Student Layer Distillation (CNN)

This notebook is fully self-contained. It defines the models, training loops, dataset generator, and student training in-place.

In [1]:
# Core imports
import os
import json
from dataclasses import dataclass

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import datasets, transforms

DEFAULT_DATA_ROOT = os.path.join(os.path.expanduser("~"), ".torch", "datasets")
print('torch:', torch.__version__)
print('dataset root:', DEFAULT_DATA_ROOT)

torch: 2.9.0+cu126
dataset root: /root/.torch/datasets


## 1) Teacher CNN

In [2]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, pool=False):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
        # Keep spatial shape identical across layers for distillation
        self.pool = nn.Identity()

    def forward(self, x):
        x = self.net(x)
        x = self.pool(x)
        return x


class TeacherNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=100, channels=None):
        super().__init__()
        if channels is None:
            channels = [64, 128, 256, 512]
        self.channels = channels

        blocks = []
        c_in = in_channels
        for i, c_out in enumerate(channels):
            blocks.append(ConvBlock(c_in, c_out, pool=False))
            c_in = c_out
        self.blocks = nn.ModuleList(blocks)

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(c_in, num_classes),
        )

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        return self.classifier(x)

    @torch.no_grad()
    def forward_with_activations(self, x):
        layer_io = []
        for block in self.blocks:
            x_in = x
            x = block(x)
            layer_io.append((x_in, x))
        logits = self.classifier(x)
        return logits, layer_io


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 2) Student CNN

In [3]:
class StudentBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class StudentNet(nn.Module):
    def __init__(self, in_channels, num_layers, max_out_channels, width=64, depth=3):
        super().__init__()
        self.in_channels = in_channels
        self.num_layers = num_layers
        self.max_out_channels = max_out_channels
        self.width = width
        self.depth = depth

        blocks = []
        c_in = in_channels + num_layers
        for _ in range(depth):
            c_out = width
            blocks.append(StudentBlock(c_in, c_out))
            c_in = c_out
        self.blocks = nn.ModuleList(blocks)
        self.head = nn.Conv2d(c_in, max_out_channels, kernel_size=1)

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        return self.head(x)

## 3) Utility: masked MSE

In [4]:
def masked_mse(pred, target, mask):
    # mask shape: (C, H, W) or (1, C, H, W)
    if mask.dim() == 3:
        mask = mask.unsqueeze(0)
    diff = (pred - target) * mask
    denom = mask.sum().clamp_min(1.0)
    return (diff.pow(2).sum() / denom).clamp_min(0.0)

## 4) CIFAR-100 loaders

In [5]:
def build_teacher_loaders(data_root, batch_size, num_workers):
    train_t = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
    ])
    test_t = transforms.Compose([transforms.ToTensor()])

    train_ds = datasets.CIFAR100(root=data_root, train=True, download=True, transform=train_t)
    test_ds = datasets.CIFAR100(root=data_root, train=False, download=True, transform=test_t)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    return train_loader, test_loader


def build_distill_loader(data_root, batch_size, num_workers):
    t = transforms.Compose([transforms.ToTensor()])
    ds = datasets.CIFAR100(root=data_root, train=True, download=True, transform=t)
    return DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)

## 5) Train Teacher

In [ ]:
@dataclass
class TeacherConfig:
    data_root: str = DEFAULT_DATA_ROOT
    out_dir: str = './runs/teacher'
    batch_size: int = 128
    epochs: int = 50
    lr: float = 0.1
    num_workers: int = 2
    channels: tuple = (64, 128, 256, 512)


def evaluate_teacher(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            logits = model(x)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.numel()
    return correct / max(total, 1)


def train_teacher(cfg: TeacherConfig):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    train_loader, test_loader = build_teacher_loaders(cfg.data_root, cfg.batch_size, cfg.num_workers)

    model = TeacherNet(num_classes=100, channels=list(cfg.channels)).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=cfg.lr, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[60, 120, 160], gamma=0.2)
    loss_fn = nn.CrossEntropyLoss()

    os.makedirs(cfg.out_dir, exist_ok=True)
    best_acc = 0.0

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = loss_fn(logits, y)
            loss.backward()
            optimizer.step()

        scheduler.step()
        acc = evaluate_teacher(model, test_loader, device)
        if acc > best_acc:
            best_acc = acc
            torch.save({'model': model.state_dict(), 'acc': acc, 'epoch': epoch}, os.path.join(cfg.out_dir, 'teacher_best.pt'))
        if epoch % 50 == 0:
            torch.save({'model': model.state_dict(), 'acc': acc, 'epoch': epoch}, os.path.join(cfg.out_dir, f'teacher_epoch_{epoch}.pt'))
        print(f'epoch={epoch} acc={acc:.4f}')

    return os.path.join(cfg.out_dir, 'teacher_best.pt')

## 6) Generate Offline Layer Dataset

In [7]:
@dataclass
class DistillDataConfig:
    data_root: str = DEFAULT_DATA_ROOT
    out_dir: str = './distill_data'
    batch_size: int = 128
    num_workers: int = 2
    teacher_ckpt: str = './runs/teacher/teacher_best.pt'
    channels: tuple = (64, 128, 256, 512)


def generate_layer_dataset(cfg: DistillDataConfig):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    loader = build_distill_loader(cfg.data_root, cfg.batch_size, cfg.num_workers)

    model = TeacherNet(num_classes=100, channels=list(cfg.channels)).to(device)
    ckpt = torch.load(cfg.teacher_ckpt, map_location=device)
    model.load_state_dict(ckpt['model'])
    model.eval()

    layer_inputs = None
    layer_outputs = None
    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device)
            _, layer_io = model.forward_with_activations(x)
            if layer_inputs is None:
                layer_inputs = [[] for _ in range(len(layer_io))]
                layer_outputs = [[] for _ in range(len(layer_io))]
            for i, (x_in, y_out) in enumerate(layer_io):
                layer_inputs[i].append(x_in.cpu())
                layer_outputs[i].append(y_out.cpu())

    os.makedirs(cfg.out_dir, exist_ok=True)
    layer_files = []
    max_in_c = 0
    max_out_c = 0
    max_h = 0
    max_w = 0

    for i in range(len(layer_inputs)):
        x = torch.cat(layer_inputs[i], dim=0)
        y = torch.cat(layer_outputs[i], dim=0)
        max_in_c = max(max_in_c, x.shape[1])
        max_out_c = max(max_out_c, y.shape[1])
        max_h = max(max_h, x.shape[2], y.shape[2])
        max_w = max(max_w, x.shape[3], y.shape[3])
        rel_path = f'layer_{i}.pt'
        torch.save({'x': x, 'y': y}, os.path.join(cfg.out_dir, rel_path))
        layer_files.append(rel_path)

    manifest = {
        'num_layers': len(layer_files),
        'max_in_channels': max_in_c,
        'max_out_channels': max_out_c,
        'max_h': max_h,
        'max_w': max_w,
        'layer_files': layer_files,
    }
    with open(os.path.join(cfg.out_dir, 'manifest.json'), 'w', encoding='utf-8') as f:
        json.dump(manifest, f, indent=2)

    return cfg.out_dir

## 7) Distillation Dataset Loader

In [8]:
class LayerDataset(Dataset):
    def __init__(self, root):
        self.root = root
        manifest_path = os.path.join(root, 'manifest.json')
        with open(manifest_path, 'r', encoding='utf-8') as f:
            self.manifest = json.load(f)

        self.num_layers = self.manifest['num_layers']
        self.max_in_channels = self.manifest['max_in_channels']
        self.max_out_channels = self.manifest['max_out_channels']
        self.max_h = self.manifest['max_h']
        self.max_w = self.manifest['max_w']
        self.layer_files = self.manifest['layer_files']

        self.layers = []
        self.index = []
        for layer_id, rel_path in enumerate(self.layer_files):
            data = torch.load(os.path.join(root, rel_path), map_location='cpu')
            x = data['x']
            y = data['y']
            self.layers.append({'x': x, 'y': y})
            for i in range(x.shape[0]):
                self.index.append((layer_id, i))

        self.layer_sizes = [layer['x'].shape[0] for layer in self.layers]
        self.weights = self._build_weights()

    def _build_weights(self):
        weights = []
        for layer_id, _ in self.index:
            weights.append(1.0 / float(self.layer_sizes[layer_id]))
        return torch.tensor(weights, dtype=torch.float)

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        layer_id, local_idx = self.index[idx]
        x = self.layers[layer_id]['x'][local_idx]
        y = self.layers[layer_id]['y'][local_idx]
        orig_out_c = y.shape[0]
        orig_h, orig_w = y.shape[1], y.shape[2]

        if x.shape[0] < self.max_in_channels:
            pad_c = self.max_in_channels - x.shape[0]
            pad = torch.zeros((pad_c, x.shape[1], x.shape[2]), dtype=x.dtype)
            x = torch.cat([x, pad], dim=0)

        if x.shape[1] < self.max_h or x.shape[2] < self.max_w:
            pad_h = self.max_h - x.shape[1]
            pad_w = self.max_w - x.shape[2]
            x = torch.nn.functional.pad(x, (0, pad_w, 0, pad_h))

        one_hot = torch.zeros((self.num_layers, x.shape[1], x.shape[2]), dtype=x.dtype)
        one_hot[layer_id, :, :] = 1.0
        x = torch.cat([x, one_hot], dim=0)

        if y.shape[1] < self.max_h or y.shape[2] < self.max_w:
            pad_h = self.max_h - y.shape[1]
            pad_w = self.max_w - y.shape[2]
            y = torch.nn.functional.pad(y, (0, pad_w, 0, pad_h))

        if y.shape[0] < self.max_out_channels:
            pad_c = self.max_out_channels - y.shape[0]
            pad = torch.zeros((pad_c, y.shape[1], y.shape[2]), dtype=y.dtype)
            y = torch.cat([y, pad], dim=0)

        channel_mask = torch.zeros((self.max_out_channels, 1, 1), dtype=y.dtype)
        channel_mask[:orig_out_c] = 1.0
        spatial_mask = torch.zeros((1, self.max_h, self.max_w), dtype=y.dtype)
        spatial_mask[:, :orig_h, :orig_w] = 1.0
        mask = channel_mask * spatial_mask

        return x, y, mask, layer_id

## 8) Student Training

In [9]:
@dataclass
class StudentConfig:
    data_root: str = './distill_data'
    out_dir: str = './runs/student'
    batch_size: int = 64
    epochs: int = 50
    lr: float = 1e-3
    num_workers: int = 2
    depth: int = 3
    base_width: int = 64
    targets: tuple = (0.1, 0.2, 0.3, 0.4, 0.5)
    teacher_channels: tuple = (64, 128, 256, 512)


def find_width_for_ratio(base_width, depth, in_ch, num_layers, max_out_ch, target_ratio, teacher_params):
    lo, hi = 8, 512
    best = None
    for _ in range(20):
        mid = (lo + hi) // 2
        model = StudentNet(in_ch, num_layers, max_out_ch, width=mid, depth=depth)
        params = count_params(model)
        ratio = params / teacher_params
        if best is None or abs(ratio - target_ratio) < abs(best[1] - target_ratio):
            best = (mid, ratio)
        if ratio < target_ratio:
            lo = mid + 1
        else:
            hi = mid - 1
    return best[0], best[1]


def train_student_variants(cfg: StudentConfig):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    dataset = LayerDataset(cfg.data_root)
    sampler = WeightedRandomSampler(dataset.weights, num_samples=len(dataset), replacement=True)
    loader = DataLoader(dataset, batch_size=cfg.batch_size, sampler=sampler, num_workers=cfg.num_workers)

    teacher = TeacherNet(num_classes=100, channels=list(cfg.teacher_channels))
    teacher_params = count_params(teacher)

    os.makedirs(cfg.out_dir, exist_ok=True)

    for target_ratio in cfg.targets:
        width, ratio = find_width_for_ratio(
            cfg.base_width, cfg.depth, dataset.max_in_channels, dataset.num_layers, dataset.max_out_channels,
            target_ratio, teacher_params
        )

        model = StudentNet(dataset.max_in_channels, dataset.num_layers, dataset.max_out_channels, width=width, depth=cfg.depth)
        model.to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)

        run_name = f'student_{int(target_ratio * 100)}pct_w{width}'
        run_dir = os.path.join(cfg.out_dir, run_name)
        os.makedirs(run_dir, exist_ok=True)

        for epoch in range(1, cfg.epochs + 1):
            model.train()
            total_loss = 0.0
            for x, y, mask, _layer_id in loader:
                x = x.to(device)
                y = y.to(device)
                mask = mask.to(device)
                optimizer.zero_grad()
                pred = model(x)
                loss = masked_mse(pred, y, mask)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()

            avg_loss = total_loss / max(len(loader), 1)
            if epoch % 10 == 0 or epoch == cfg.epochs:
                torch.save({
                    'model': model.state_dict(),
                    'epoch': epoch,
                    'loss': avg_loss,
                    'ratio': ratio
                }, os.path.join(run_dir, f'student_epoch_{epoch}.pt'))
            print(f'ratio_target={target_ratio:.2f} ratio_actual={ratio:.4f} epoch={epoch} loss={avg_loss:.6f}')

## 9) Run the Pipeline

In [11]:
# Adjust configs if needed
teacher_cfg = TeacherConfig()
distill_cfg = DistillDataConfig()
student_cfg = StudentConfig()


In [12]:

# 1) Train teacher
best_ckpt = train_teacher(teacher_cfg)


epoch=1 acc=0.0547
epoch=2 acc=0.1285
epoch=3 acc=0.1376
epoch=4 acc=0.1459
epoch=5 acc=0.2172
epoch=6 acc=0.2681
epoch=7 acc=0.3308
epoch=8 acc=0.2931
epoch=9 acc=0.3125
epoch=10 acc=0.3201
epoch=11 acc=0.3114
epoch=12 acc=0.3424
epoch=13 acc=0.3581
epoch=14 acc=0.2416
epoch=15 acc=0.3703
epoch=16 acc=0.3582
epoch=17 acc=0.3884
epoch=18 acc=0.3109
epoch=19 acc=0.3810
epoch=20 acc=0.2847
epoch=21 acc=0.4352
epoch=22 acc=0.2862
epoch=23 acc=0.4648
epoch=24 acc=0.4581
epoch=25 acc=0.4329
epoch=26 acc=0.4435
epoch=27 acc=0.3998
epoch=28 acc=0.4436
epoch=29 acc=0.4079
epoch=30 acc=0.4278
epoch=31 acc=0.3565
epoch=32 acc=0.4750
epoch=33 acc=0.3075
epoch=34 acc=0.4302
epoch=35 acc=0.3903
epoch=36 acc=0.4649
epoch=37 acc=0.3915
epoch=38 acc=0.4914
epoch=39 acc=0.5175
epoch=40 acc=0.4434
epoch=41 acc=0.2814
epoch=42 acc=0.4769
epoch=43 acc=0.4710
epoch=44 acc=0.3925
epoch=45 acc=0.2777
epoch=46 acc=0.4964
epoch=47 acc=0.4874
epoch=48 acc=0.4297
epoch=49 acc=0.3824
epoch=50 acc=0.5536
epoch=51 

: 

In [1]:

# 2) Generate distillation dataset
distill_cfg.teacher_ckpt = best_ckpt
generate_layer_dataset(distill_cfg)


NameError: name 'best_ckpt' is not defined

In [ ]:

# 3) Train student variants
train_student_variants(student_cfg)